# spectral-trust quickstart

[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/vcnoel/spectral-trust/blob/main/examples/quickstart.ipynb)

This notebook runs `spectral-trust` on a small open model, `Qwen/Qwen3-0.6B`:

1. `SpectralModel.run_with_spectra` and the `SpectralCache` it returns;
2. one head, one layer, and a long table;
3. the interactive views;
4. a subset of heads, and a batch of texts;
5. the same measurements from a Hugging Face output, a TransformerLens cache
   and an nnsight trace.

A CPU is enough. The first run downloads the model (about 1.5 GB) from the
Hugging Face Hub; later runs read it from the local cache.

In [1]:
# Install spectral-trust if it is missing (on Colab, for example).
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("spectral_trust") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spectral-trust>=0.4.0"])

os.environ.setdefault("TQDM_DISABLE", "1")  # no progress bars in the outputs
import spectral_trust as st

print("spectral-trust", st.__version__)

spectral-trust 0.4.0rc1


## Load a model and run it

`SpectralModel.from_pretrained` loads the causal LM with eager attention, the
implementation that returns attention weights. `run_with_spectra` runs it
once and measures every layer.

In [2]:
import transformers

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

model = st.SpectralModel.from_pretrained("Qwen/Qwen3-0.6B")
text = "The cat sat on the mat because it was warm."
cache = model.run_with_spectra(text)
print(model)
print(cache)
print(cache.tokens)

SpectralModel('Qwen/Qwen3-0.6B', 28 layers, 16 heads, normalization='sym')
SpectralCache('Qwen/Qwen3-0.6B', 11 tokens, layers 0-27, 16 heads, metrics: attn=5, resid=4, graph=10)
['The', ' cat', ' sat', ' on', ' the', ' mat', ' because', ' it', ' was', ' warm', '.']


## The cache

Keys follow TransformerLens' naming. `blocks.{l}.attn.*` holds five
attention-only metrics of each head's graph, `blocks.{l}.resid.*` four
metrics of the residual stream on each head's graph, and `blocks.{l}.graph.*`
every metric of the head-averaged graph. Each metric is tagged with its
family: *attention* metrics read the attention weights only, *hybrid* ones
also read the hidden states.

In [3]:
print(cache["blocks.12.attn.fiedler_value"].round(3))   # one value per head
print(round(cache["blocks.12.graph.hfer"], 4))         # the head-averaged graph
print({name: cache.families[name] for name in ("attn.fiedler_value", "resid.hfer", "graph.hfer")})
cache.layer(12).head(3)

[0.898 0.7   0.503 0.671 0.538 0.537 0.479 0.806 0.757 0.824 0.632 0.669
 0.631 0.734 0.646 0.802]
0.5577
{'attn.fiedler_value': 'attention', 'resid.hfer': 'hybrid', 'graph.hfer': 'hybrid'}


HeadSpectra(layer 12, head 3)
  attn.fiedler_value           0.6710
  attn.connectivity_ratio      0.4614
  attn.spectral_entropy_norm   0.9512
  attn.hfer                    0.6710
  attn.spectral_radius         1.4541
  resid.energy                 33221092.0000
  resid.smoothness_index       0.7688
  resid.spectral_entropy       0.9702
  resid.hfer                   0.4804

`to_dataframe` gives one row per (layer, head, metric). `head` is missing for
the head-averaged `graph` metrics.

In [4]:
df = cache.to_dataframe()
print(len(df), "rows")
per_layer = df[df.metric.isin(["graph.fiedler_value", "graph.hfer"])].pivot(
    index="layer", columns="metric", values="value")
per_layer.round(4).head(8)

4312 rows


metric,graph.fiedler_value,graph.hfer
layer,,
0,0.2554,0.0991
1,0.4718,0.0726
2,0.4465,0.0147
3,0.8769,0.5123
4,0.6625,0.5562
5,0.7384,0.5723
6,0.6319,0.5820
7,0.6389,0.5817


## Views

Each view is self-contained SVG: it renders here, and `.save("view.html")`
writes a page that opens offline. Hover the cells, nodes, edges and points
for their values.

In [5]:
cache.heatmap("fiedler_value")

<heatmap view: Heatmap: attn.fiedler_value>

In [6]:
import numpy as np

fiedler = cache.metric("fiedler_value")                        # [layers, heads]
row, head = np.unravel_index(np.nanargmin(fiedler), fiedler.shape)
layer = cache.layers[row]
print(f"smallest Fiedler value: layer {layer}, head {head}, {fiedler[row, head]:.4f}")
cache.graph(layer, head=int(head))

smallest Fiedler value: layer 6, head 7, 0.0000


<graph view: Attention graph: layer 6, head 7>

In [7]:
cache.spectrum(layer, head=int(head))

<spectrum view: Spectrum: layer 6, head 7>

## A subset of heads, and a batch

`heads=` computes only some heads. The scores that choose them come from
outside the library; here, as an example, each head's mean attention to the
first token, read from the raw patterns the cache kept.

In [8]:
scores = np.stack([cache[f"blocks.{l}.attn.pattern"][:, :, 0].mean(-1) for l in cache.layers])
top = st.heads_from_scores(scores, k=10)                       # [(layer, head), ...]
subset = model.run_with_spectra(text, heads=top, metrics=["attn"])
print(subset.heads)
print("subset equals the full computation:",
      all(np.allclose(subset[f"blocks.{l}.attn.fiedler_value"][h],
                      cache[f"blocks.{l}.attn.fiedler_value"][h]) for l, h in top))

{0: [], 1: [], 2: [], 3: [], 4: [9], 5: [], 6: [], 7: [], 8: [], 9: [], 10: [], 11: [], 12: [], 13: [], 14: [], 15: [], 16: [11, 14, 15], 17: [], 18: [], 19: [4], 20: [3], 21: [8, 9], 22: [0, 3], 23: [], 24: [], 25: [], 26: [], 27: []}
subset equals the full computation: True


In [9]:
texts = ["The capital of France is Paris.", "The capital of France is Rome."]
batch = model.run_with_spectra(texts)
print(batch)
table = batch.to_dataframe(metrics=["graph.fiedler_value"]).pivot(
    index="layer", columns="text_id", values="value")
table.round(4).tail(6)

SpectralCache('Qwen/Qwen3-0.6B', 2 texts of 7 tokens, layers 0-27, 16 heads, metrics: attn=5, resid=4, graph=10)


text_id,0,1
layer,,
22,0.8439,0.8411
23,0.7683,0.7658
24,0.8399,0.8364
25,0.8732,0.8705
26,0.7848,0.7844
27,0.7566,0.7461


## From a Hugging Face output

The same computation runs on tensors you already have. A forward pass with
`output_attentions=True`:

In [10]:
import torch

ids = torch.tensor([model.tokenizer(text)["input_ids"]])
with torch.no_grad():
    out = model.model(input_ids=ids, output_attentions=True, output_hidden_states=True)
from_hf = st.from_hf_outputs(out, model.model.config, ids, tokenizer=model.tokenizer)
print("largest difference to run_with_spectra:",
      max(float(np.nanmax(np.abs(np.asarray(from_hf[k], float) - np.asarray(cache[k], float))))
          for k in cache.keys()))

largest difference to run_with_spectra: 0.0


## With TransformerLens or nnsight

`from_transformer_lens` reads a TransformerLens cache and `from_nnsight`
tensors saved in an nnsight trace, after any intervention. Both are optional
extras (`pip install "spectral-trust[tl,nnsight]"`); the cells below run when
they are installed. Both reuse the model loaded above.

In [11]:
if importlib.util.find_spec("nnsight") is None:
    print("pip install 'spectral-trust[nnsight]' to run this cell")
else:
    import warnings
    from nnsight import LanguageModel

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lm = LanguageModel(model.model, tokenizer=model.tokenizer)
    attn, resid = [], []
    with lm.trace(text):
        for block in lm.model.layers:
            attn.append(block.self_attn.output[1].save())   # attention weights
            resid.append(block.output.save())               # block output
        final = lm.model.norm.output.save()                  # the last hidden state
    from_nn = st.from_nnsight(attn, resid, model=lm, tokens=text, final_norm=final)
    print(from_nn)
    print("largest difference to run_with_spectra:",
          max(float(np.nanmax(np.abs(np.asarray(from_nn[k], float) - np.asarray(cache[k], float))))
              for k in cache.keys() if not k.endswith("pattern")))

SpectralCache('Qwen/Qwen3-0.6B', 11 tokens, layers 0-27, 16 heads, metrics: attn=5, resid=4, graph=10)
largest difference to run_with_spectra: 0.0


In [12]:
if importlib.util.find_spec("transformer_lens") is None:
    print("pip install 'spectral-trust[tl]' to run this cell")
else:
    from transformer_lens.model_bridge import TransformerBridge

    bridge = TransformerBridge.boot_transformers("Qwen/Qwen3-0.6B", hf_model=model.model,
                                                 device="cpu")
    _, tl_cache = bridge.run_with_cache(ids)
    from_tl = st.from_transformer_lens(tl_cache, bridge, ids)
    print(from_tl)
    print("fiedler_value, layer 12:", from_tl["blocks.12.attn.fiedler_value"][:4].round(4),
          "run_with_spectra:", cache["blocks.12.attn.fiedler_value"][:4].round(4))
    del bridge, tl_cache

SpectralCache('Qwen/Qwen3-0.6B', 11 tokens, layers 0-27, 16 heads, metrics: attn=5, resid=4, graph=10)
fiedler_value, layer 12: [0.8981 0.6999 0.5029 0.671 ] run_with_spectra: [0.8981 0.6999 0.5029 0.671 ]
